# Lab 02: Multilayer Perceptrons, Activations, and Losses

            **Duration:** 3 hours  
            **Lecture alignment:** Week 2 — Feedforward neural networks  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Build a multilayer perceptron with `nn.Module`.
- Relate activation and loss choices to task behavior.
- Compare models with controlled seeds, data, and training budgets.

            ## Three-hour activity plan

            - 0–25 min: generate, split, and visualize data
- 25–65 min: activation/loss investigation
- 65–120 min: implement and train MLP variants
- 120–160 min: decision boundaries and metric comparison
- 160–180 min: checks and evidence-based recommendation


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20262
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_02")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_02"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 2, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Which activation will reach the best test accuracy under the same budget, and why might the ranking change with a deeper network?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Generate two moons and inspect activation functions


In [ ]:
def make_moons(n=700, noise=.10):
    half = n // 2
    t = torch.rand(half) * math.pi
    outer = torch.stack([torch.cos(t), torch.sin(t)], 1)
    inner = torch.stack([1 - torch.cos(t), 0.45 - torch.sin(t)], 1)
    X = torch.cat([outer, inner]) + noise * torch.randn(n, 2)
    y = torch.cat([torch.zeros(half), torch.ones(half)]).long()
    order = torch.randperm(n)
    return X[order], y[order]

X, y = make_moons(600 if FAST_MODE else 2400)
split = int(.75 * len(X)); X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
grid = torch.linspace(-4, 4, 300)
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, fn in {"ReLU": F.relu, "sigmoid": torch.sigmoid, "tanh": torch.tanh, "GELU": F.gelu}.items():
    ax.plot(grid, fn(grid), label=name)
ax.legend(ncol=4); ax.set_title("Activation functions"); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "activations.png", dpi=150); plt.show()


## Activity 2 — Controlled activation comparison


In [ ]:
class MLP(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 24), activation, nn.Linear(24, 24), activation, nn.Linear(24, 2))
    def forward(self, x): return self.net(x)

def train_mlp(activation, epochs=45):
    torch.manual_seed(SEED)
    model = MLP(activation).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=.025)
    history = []
    for _ in range(epochs):
        opt.zero_grad(); logits = model(X_train.to(DEVICE))
        loss = F.cross_entropy(logits, y_train.to(DEVICE)); loss.backward(); opt.step()
        history.append(loss.item())
    with torch.no_grad():
        acc = (model(X_test.to(DEVICE)).argmax(1).cpu() == y_test).float().mean().item()
    return model, history, acc

results = {}
activations = {"ReLU": nn.ReLU(), "Tanh": nn.Tanh(), "GELU": nn.GELU()}
for name, activation in activations.items():
    results[name] = train_mlp(activation, 35 if FAST_MODE else 100)
print({name: round(value[2], 3) for name, value in results.items()})

best_name = max(results, key=lambda name: results[name][2])
best_model = results[best_name][0]
xx, yy = torch.meshgrid(torch.linspace(X[:,0].min()-.3, X[:,0].max()+.3, 120),
                        torch.linspace(X[:,1].min()-.3, X[:,1].max()+.3, 120), indexing="xy")
points = torch.stack([xx.flatten(), yy.flatten()], 1)
with torch.no_grad(): zz = best_model(points.to(DEVICE)).argmax(1).cpu().reshape(xx.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, (_, history, _) in results.items(): axes[0].plot(history, label=name)
axes[0].legend(); axes[0].set(title="Loss comparison", xlabel="epoch")
axes[1].contourf(xx, yy, zz, alpha=.25); axes[1].scatter(X_test[:,0], X_test[:,1], c=y_test, s=12, cmap="coolwarm")
axes[1].set_title(f"{best_name} decision boundary")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "mlp_comparison.png", dpi=150); plt.show()


## Automated checks


In [ ]:
assert X.shape[1] == 2 and set(y.tolist()) == {0, 1}
assert all(np.isfinite(history).all() for _, history, _ in results.values())
assert max(value[2] for value in results.values()) > .78
assert (ARTIFACT_DIR / "mlp_comparison.png").exists()
print("All Lab 02 checks passed.")


## Deliverables

                - Activation plot
- Executable MLP and decision-boundary plot
- Controlled comparison table and architecture recommendation

                Submit the executed notebook and the files created in `/content/artifacts/lab_02/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    outlier_x = torch.linspace(-2, 2, 100).unsqueeze(1)
    outlier_y = 2*outlier_x + .1*torch.randn_like(outlier_x); outlier_y[::10] += 5
    print("Extension: fit identical regressors with MSE and Huber loss and compare sensitivity to outliers.")
else:
    print("Extension disabled: compare MSE and Huber loss on regression with outliers.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
